# Fine-tuning XLM-RoBERTa para NER en Dataset Próstata

**Objetivo**: Entrenar modelo XLM-RoBERTa (multilingüe) para reconocimiento de entidades nombradas en textos clínicos de cáncer de próstata.

**Modelo**: `xlm-roberta-base` (XLM-R)

**Dataset**: Próstata (10 tipos de entidades clínicas)

**Configuración**:
- Batch sizes: 8, 16, 32
- Épocas: 6-10 con Early Stopping
- Métricas: F1-score, Precision, Recall
- Objetivo: Superar F1 72.00% (BiLSTM baseline)

---

## 📋 Instrucciones para Kaggle

### 1. Configuración del Notebook
- **Accelerator**: GPU T4 x2
- **Internet**: ON (para descargar modelos y subir a HuggingFace)

### 2. Subir Dataset
Crear carpeta `prostata/` y subir:
- `training.bio`
- `validation_cleaned.bio`
- `testing_cleaned.bio`

### 3. Token de HuggingFace
- Crear token en: https://huggingface.co/settings/tokens
- Permisos: Write
- Usar en celda de login

## 1. Instalación de Librerías

In [171]:
# Instalar librerías (separar evaluate para evitar conflictos)
!pip install transformers==4.30.0 datasets huggingface_hub -q
!pip install evaluate seqeval -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [172]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA memory: 15.64 GB


## 2. Login a HuggingFace

In [ ]:
from huggingface_hub import login

# IMPORTANTE: Reemplazar con tu token de HuggingFace
login("token huggingFace Aquí")

## 3. Configuración

In [174]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Desactivar W&B
os.environ["WANDB_DISABLED"] = "true"

# Configuración del modelo
MODEL_CHECKPOINT = "xlm-roberta-base"  # XLM-RoBERTa
MAX_LENGTH = 128

# Configuración de entrenamiento
BATCH_SIZE = 16  # Probar con 8, 16, 32
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5

# Rutas del dataset (ajustar según estructura de Kaggle)
DATA_DIR = Path("/kaggle/input/datasets/julianquimbayocastro/prostata")  # o "./prostata" si subes directamente
OUTPUT_DIR = f"./xlmr-ner-prostata-batch{BATCH_SIZE}"

RUTAS_ARCHIVOS = {
    "train": DATA_DIR / "training.bio",
    "validation": DATA_DIR / "validation_cleaned.bio",
    "test": DATA_DIR / "testing_cleaned.bio"
}

# Nombre del modelo en HuggingFace
HF_MODEL_NAME = "Lucyan85/xlmr-ner-prostata"

print(f"✅ Configuración:")
print(f"  Modelo base: {MODEL_CHECKPOINT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Épocas máximas: {NUM_EPOCHS}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  HuggingFace: {HF_MODEL_NAME}")

✅ Configuración:
  Modelo base: xlm-roberta-base
  Batch size: 16
  Épocas máximas: 10
  Output: ./xlmr-ner-prostata-batch16
  HuggingFace: Lucyan85/xlmr-ner-prostata


## 4. Carga del Dataset

In [175]:
def leer_bio_file(ruta):
    """
    Lee archivo en formato BIO (token\tlabel).
    Retorna diccionario con listas de tokens y etiquetas.
    """
    datos = {"tokens": [], "ner_tags": []}
    tokens, tags = [], []
    
    with open(ruta, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                # Línea vacía = fin de oración
                if tokens:
                    datos["tokens"].append(tokens)
                    datos["ner_tags"].append(tags)
                    tokens, tags = [], []
            else:
                # Línea con datos
                partes = line.split('\t')
                if len(partes) == 2:
                    token, label = partes
                    tokens.append(token)
                    tags.append(label)
        
        # Última oración
        if tokens:
            datos["tokens"].append(tokens)
            datos["ner_tags"].append(tags)
    
    return datos

# Cargar datasets
print("📂 Cargando datasets...")
train_data = leer_bio_file(RUTAS_ARCHIVOS["train"])
val_data = leer_bio_file(RUTAS_ARCHIVOS["validation"])
test_data = leer_bio_file(RUTAS_ARCHIVOS["test"])

print(f"\n✅ Datasets cargados:")
print(f"  Train: {len(train_data['tokens']):,} oraciones")
print(f"  Val: {len(val_data['tokens']):,} oraciones")
print(f"  Test: {len(test_data['tokens']):,} oraciones")

📂 Cargando datasets...

✅ Datasets cargados:
  Train: 3,106 oraciones
  Val: 929 oraciones
  Test: 991 oraciones


## 5. Detección y Mapeo de Etiquetas

In [176]:
def detectar_etiquetas(rutas_dict):
    """
    Detecta todas las etiquetas únicas en los archivos.
    Retorna lista ordenada: etiquetas específicas + 'O' al final.
    """
    todas = set()
    
    for ruta in rutas_dict.values():
        with open(ruta, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and '\t' in line:
                    partes = line.split('\t')
                    if len(partes) == 2:
                        todas.add(partes[1])
    
    # Ordenar: etiquetas específicas alfabéticamente, 'O' al final
    etiquetas = sorted(todas - {"O"}) + ["O"]
    return etiquetas

# Detectar etiquetas
label_list = detectar_etiquetas(RUTAS_ARCHIVOS)
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"\n🏷️ Etiquetas detectadas ({len(label_list)}):")
for i, label in enumerate(label_list):
    print(f"  {i}: {label}")

# Guardar mapeo de etiquetas
import json
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(OUTPUT_DIR) / "labels.json", "w") as f:
    json.dump(label_list, f, indent=2)
print(f"\n✅ Etiquetas guardadas en: {OUTPUT_DIR}/labels.json")


🏷️ Etiquetas detectadas (23):
  0: ,
  1: 0
  2: B-BIOMARCADOR
  3: B-CANCER
  4: B-CIRUGIA
  5: B-DOSIS
  6: B-EDAD
  7: B-FECHA
  8: B-GLEASON
  9: B-MEDICAMENTO
  10: B-TNM
  11: B-TRATAMIENTO
  12: I-BIOMARCADOR
  13: I-CANCER
  14: I-CIRUGIA
  15: I-DOSIS
  16: I-EDAD
  17: I-FECHA
  18: I-GLEASON
  19: I-MEDICAMENTO
  20: I-TNM
  21: I-TRATAMIENTO
  22: O

✅ Etiquetas guardadas en: ./xlmr-ner-prostata-batch16/labels.json


## 6. Creación de Dataset con HuggingFace

In [177]:
from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel

# Convertir etiquetas de texto a IDs
def convertir_tags_a_ids(datos, label2id):
    datos_ids = {"tokens": [], "ner_tags": []}
    for tokens, tags in zip(datos["tokens"], datos["ner_tags"]):
        datos_ids["tokens"].append(tokens)
        datos_ids["ner_tags"].append([label2id[tag] for tag in tags])
    return datos_ids

train_data_ids = convertir_tags_a_ids(train_data, label2id)
val_data_ids = convertir_tags_a_ids(val_data, label2id)
test_data_ids = convertir_tags_a_ids(test_data, label2id)

# Definir features
features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=label_list))
})

# Crear datasets
dataset = DatasetDict({
    "train": Dataset.from_dict(train_data_ids, features=features),
    "validation": Dataset.from_dict(val_data_ids, features=features),
    "test": Dataset.from_dict(test_data_ids, features=features)
})

print("\n✅ Dataset creado:")
print(dataset)
print("\n📄 Ejemplo de oración:")
print(f"  Tokens: {dataset['train'][0]['tokens'][:10]}...")
print(f"  Tags: {[id2label[tag] for tag in dataset['train'][0]['ner_tags'][:10]]}...")


✅ Dataset creado:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3106
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 929
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 991
    })
})

📄 Ejemplo de oración:
  Tokens: ['Paciente', 'de', '72', 'años', ',', 'con', 'antecedentes', 'médicos', 'de', 'HTA']...
  Tags: ['O', 'O', 'B-EDAD', 'I-EDAD', 'O', 'O', 'O', 'O', 'O', 'O']...


## 7. Tokenización

In [178]:
from transformers import AutoTokenizer

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✅ Tokenizer cargado: {MODEL_CHECKPOINT}")

def tokenize_and_align_labels(examples):
    """
    Tokeniza y alinea las etiquetas con los tokens del tokenizer.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        padding=False  # Padding dinámico con DataCollator
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                # Tokens especiales ([CLS], [SEP], [PAD])
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Primera subpalabra del token
                label_ids.append(label[word_idx])
            else:
                # Subpalabras siguientes del mismo token
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenizar datasets
print("\n🔤 Tokenizando datasets...")
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("✅ Tokenización completada")
print(tokenized_datasets)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer cargado: xlm-roberta-base

🔤 Tokenizando datasets...


Map:   0%|          | 0/3106 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

Map:   0%|          | 0/991 [00:00<?, ? examples/s]

✅ Tokenización completada
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3106
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 929
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 991
    })
})


## 8. Preparación del Modelo

In [179]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

# Cargar modelo
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Modelo cargado: {MODEL_CHECKPOINT}")
print(f"  Parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Etiquetas: {len(label_list)}")

# Data collator (padding dinámico)
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modelo cargado: xlm-roberta-base
  Parámetros: 277,470,743
  Etiquetas: 23


## 9. Métricas de Evaluación

In [180]:
import numpy as np

# Intentar cargar evaluate, si falla usar seqeval directamente
try:
    from evaluate import load
    seqeval_metric = load("seqeval")
    use_evaluate = True
    print("✅ Usando evaluate.load('seqeval')")
except (ImportError, Exception) as e:
    # Fallback: usar seqeval directamente
    from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score
    use_evaluate = False
    print("⚠️ Usando seqeval directamente (sin evaluate)")
    print(f"   Razón: {e}")

def compute_metrics(p):
    """
    Calcula métricas F1, Precision, Recall usando seqeval.
    """
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Eliminar tokens ignorados (-100)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    if use_evaluate:
        # Usar evaluate.load("seqeval")
        results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]
        }
    else:
        # Fallback: usar seqeval directamente
        return {
            "precision": precision_score(true_labels, true_predictions),
            "recall": recall_score(true_labels, true_predictions),
            "f1": f1_score(true_labels, true_predictions),
            "accuracy": accuracy_score(true_labels, true_predictions)
        }

print("✅ Métricas configuradas")

✅ Usando evaluate.load('seqeval')
✅ Métricas configuradas


## 10. Entrenamiento

In [181]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# Calcular warmup_steps (10% del total de pasos)
total_steps = (len(tokenized_datasets["train"]) // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Entrenamiento
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    
    # Evaluación
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Optimización
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=1,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    
    # Logging
    logging_steps=50,
    report_to="none",
    
    # Guardado
    save_total_limit=2,
    push_to_hub=False
)

# Crear Trainer (sin tokenizer - no es necesario si tenemos data_collator)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n🚀 Iniciando entrenamiento...\n")
print(f"Configuración:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Épocas: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Early stopping: 3 épocas sin mejora")
print(f"  FP16: {training_args.fp16}")
print()


🚀 Iniciando entrenamiento...

Configuración:
  Batch size: 16
  Épocas: 10
  Learning rate: 2e-05
  Warmup steps: 194
  Early stopping: 3 épocas sin mejora
  FP16: True



In [182]:
# Entrenar
train_result = trainer.train()

# Métricas de entrenamiento
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

# Guardar modelo y tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Modelo y tokenizer guardados en: {OUTPUT_DIR}")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,5.841031,1.602516,0.000000,0.000000,0.000000,0.807945
2,1.521273,0.532948,0.665786,0.656678,0.661200,0.946691
3,0.481179,0.212631,0.792662,0.886645,0.837023,0.971282
4,0.178641,0.089135,0.914947,0.953094,0.933631,0.988991
5,0.113091,0.056901,0.963494,0.962866,0.963180,0.994316
6,0.090836,0.048803,0.956941,0.970033,0.963442,0.994137
7,0.064447,0.045049,0.969599,0.976547,0.973061,0.994855
8,0.048160,0.042091,0.967287,0.982410,0.974790,0.995393
9,0.043887,0.038296,0.981795,0.983713,0.982753,0.996590
10,0.037884,0.037071,0.980507,0.983062,0.981783,0.996410


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

***** train metrics *****
  epoch                    =       10.0
  total_flos               =  1714948GF
  train_loss               =      0.631
  train_runtime            = 0:12:08.47
  train_samples_per_second =     42.637
  train_steps_per_second   =      1.345


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Modelo y tokenizer guardados en: ./xlmr-ner-prostata-batch16


## 11. Evaluación en Test

In [183]:
# Evaluar en test
print("\n📊 Evaluando en conjunto de prueba...\n")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n🎯 Resultados en Test:")
print(f"  F1-score:  {test_results['eval_f1']*100:.2f}%")
print(f"  Precision: {test_results['eval_precision']*100:.2f}%")
print(f"  Recall:    {test_results['eval_recall']*100:.2f}%")
print(f"  Accuracy:  {test_results['eval_accuracy']*100:.2f}%")

# Guardar resultados
trainer.log_metrics("test", test_results)
trainer.save_metrics("test", test_results)

# Comparación con BiLSTM
bilstm_f1 = 72.00
improvement = test_results['eval_f1']*100 - bilstm_f1
print(f"\n📈 Comparación con BiLSTM (CoNLL-2002):")
print(f"  BiLSTM F1:  {bilstm_f1:.2f}%")
print(f"  XLM-R F1:   {test_results['eval_f1']*100:.2f}%")
print(f"  Diferencia: {improvement:+.2f} puntos")


📊 Evaluando en conjunto de prueba...




🎯 Resultados en Test:
  F1-score:  96.70%
  Precision: 96.41%
  Recall:    97.00%
  Accuracy:  99.37%
***** test metrics *****
  epoch                   =       10.0
  eval_accuracy           =     0.9937
  eval_f1                 =      0.967
  eval_loss               =     0.0606
  eval_precision          =     0.9641
  eval_recall             =       0.97
  eval_runtime            = 0:00:06.50
  eval_samples_per_second =    152.235
  eval_steps_per_second   =      4.762

📈 Comparación con BiLSTM (CoNLL-2002):
  BiLSTM F1:  72.00%
  XLM-R F1:   96.70%
  Diferencia: +24.70 puntos


## 12. Publicar en HuggingFace

In [184]:
# Crear model card
model_card = f"""---
language: es
tags:
- token-classification
- ner
- medical
- spanish
- multilingual
- prostate-cancer
datasets:
- prostata
metrics:
- f1
- precision
- recall
model-index:
- name: XLM-RoBERTa NER Próstata
  results:
  - task:
      type: token-classification
      name: Named Entity Recognition
    metrics:
    - type: f1
      value: {test_results['eval_f1']:.4f}
      name: F1-score
    - type: precision
      value: {test_results['eval_precision']:.4f}
      name: Precision
    - type: recall
      value: {test_results['eval_recall']:.4f}
      name: Recall
---

# XLM-RoBERTa Fine-tuned for NER on Prostate Cancer Clinical Texts

## Descripción

Modelo XLM-RoBERTa (multilingüe) afinado para reconocimiento de entidades nombradas (NER) en textos clínicos sobre cáncer de próstata.

**Modelo base**: `xlm-roberta-base`

## Entidades Reconocidas

El modelo identifica 10 tipos de entidades clínicas:

1. **EDAD**: Edad del paciente
2. **BIOMARCADOR**: PSA, marcadores biológicos
3. **CANCER**: Tipos de cáncer
4. **GLEASON**: Score de Gleason
5. **TNM**: Clasificación TNM
6. **TRATAMIENTO**: Tratamientos médicos
7. **MEDICAMENTO**: Medicamentos
8. **DOSIS**: Dosis de medicamentos
9. **CIRUGIA**: Procedimientos quirúrgicos
10. **FECHA**: Fechas de eventos médicos

## Rendimiento

| Métrica | Valor |
|---------|-------|
| F1-score | {test_results['eval_f1']*100:.2f}% |
| Precision | {test_results['eval_precision']*100:.2f}% |
| Recall | {test_results['eval_recall']*100:.2f}% |
| Accuracy | {test_results['eval_accuracy']*100:.2f}% |

## Uso

```python
from transformers import pipeline

# Cargar modelo
ner_pipeline = pipeline(
    "token-classification",
    model="{HF_MODEL_NAME}",
    aggregation_strategy="simple"
)

# Ejemplo
texto = '''Paciente masculino de 72 años con adenocarcinoma de próstata. 
Gleason 3+3, PSA 9.9 ng/dL.'''

resultados = ner_pipeline(texto)
for entidad in resultados:
    print(f"{{entidad['entity_group']}}: {{entidad['word']}}")
```

## Entrenamiento

- **Batch size**: {BATCH_SIZE}
- **Learning rate**: {LEARNING_RATE}
- **Épocas**: {NUM_EPOCHS}
- **Max length**: {MAX_LENGTH}
- **Early stopping**: 3 épocas

## Comparación con BETO

XLM-RoBERTa es un modelo multilingüe entrenado en 100 idiomas, mientras que BETO está especializado en español. 
Este modelo puede generalizar mejor en textos con términos médicos en diferentes idiomas.

## Autor

Lucyan85 - Maestría en Ingeniería de Sistemas y Computación, Universidad Nacional de Colombia
"""

# Guardar README
with open(Path(OUTPUT_DIR) / "README.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print("✅ Model card creado")

✅ Model card creado


In [185]:
# Subir a HuggingFace Hub
print(f"\n📤 Subiendo modelo a HuggingFace: {HF_MODEL_NAME}\n")

# Subir modelo y tokenizer (método compatible con transformers 4.30.0)
print("📦 Subiendo modelo...")
model.push_to_hub(
    HF_MODEL_NAME,
    commit_message=f"XLM-R NER Próstata - F1: {test_results['eval_f1']*100:.2f}% (batch_size={BATCH_SIZE})"
)

print("📦 Subiendo tokenizer...")
tokenizer.push_to_hub(
    HF_MODEL_NAME,
    commit_message=f"XLM-R NER Próstata - F1: {test_results['eval_f1']*100:.2f}% (batch_size={BATCH_SIZE})"
)

# Subir README (model card)
from huggingface_hub import HfApi, upload_file
api = HfApi()
print("📦 Subiendo README...")
upload_file(
    path_or_fileobj=Path(OUTPUT_DIR) / "README.md",
    path_in_repo="README.md",
    repo_id=HF_MODEL_NAME,
    repo_type="model"
)

print(f"\n✅ Modelo publicado en: https://huggingface.co/{HF_MODEL_NAME}")


📤 Subiendo modelo a HuggingFace: Lucyan85/xlmr-ner-prostata

📦 Subiendo modelo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📦 Subiendo tokenizer...


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📦 Subiendo README...

✅ Modelo publicado en: https://huggingface.co/Lucyan85/xlmr-ner-prostata


## 13. Prueba del Modelo Publicado

In [186]:
from transformers import pipeline

# Cargar modelo desde HuggingFace
print(f"\n🧪 Probando modelo publicado: {HF_MODEL_NAME}\n")
ner_pipeline = pipeline(
    "token-classification",
    model=HF_MODEL_NAME,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

# Texto de prueba
texto_prueba = """Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial. 
Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. 
El diagnóstico histológico reveló un adenocarcinoma de próstata. 
El puntaje de Gleason reportado fue 3+3, lo que indica un cáncer de bajo grado. 
El PSA más alto registrado fue de 9,9 ng/dL."""

print("📄 Texto de prueba:")
print(texto_prueba)
print("\n🔍 Entidades detectadas:\n")

resultados = ner_pipeline(texto_prueba)
for entidad in resultados:
    print(f"  [{entidad['entity_group']}] {entidad['word']} (score: {entidad['score']:.2f})")

print(f"\n✅ Total de entidades detectadas: {len(resultados)}")


🧪 Probando modelo publicado: Lucyan85/xlmr-ner-prostata



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

Invalid model-index. Not loading eval results into CardData.


📄 Texto de prueba:
Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial. 
Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. 
El diagnóstico histológico reveló un adenocarcinoma de próstata. 
El puntaje de Gleason reportado fue 3+3, lo que indica un cáncer de bajo grado. 
El PSA más alto registrado fue de 9,9 ng/dL.

🔍 Entidades detectadas:

  [EDAD] 72 años (score: 0.99)
  [CANCER] a (score: 1.00)
  [CANCER] denocarcinoma de próstata (score: 0.99)
  [GLEASON] G (score: 1.00)
  [GLEASON] leason report (score: 0.63)
  [GLEASON] 3+3 (score: 0.75)
  [CANCER] cáncer (score: 0.94)
  [BIOMARCADOR] PSA más alto registrado (score: 0.97)
  [BIOMARCADOR] de 9,9 ng/dL (score: 0.90)

✅ Total de entidades detectadas: 9


## 14. Resumen Final

In [187]:
print("="*80)
print("RESUMEN DEL ENTRENAMIENTO")
print("="*80)
print(f"\nModelo: XLM-RoBERTa (xlm-roberta-base)")
print(f"Dataset: Próstata (NER clínico)")
print(f"Entidades: {len(label_list)} etiquetas (10 tipos + O)")
print(f"\nConfiguración:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Épocas entrenadas: {train_result.metrics.get('epoch', NUM_EPOCHS):.0f}")
print(f"\n🎯 Resultados en Test:")
print(f"  F1-score:  {test_results['eval_f1']*100:.2f}%")
print(f"  Precision: {test_results['eval_precision']*100:.2f}%")
print(f"  Recall:    {test_results['eval_recall']*100:.2f}%")
print(f"\n📊 Comparación:")
print(f"  BiLSTM (CoNLL-2002): {bilstm_f1:.2f}%")
print(f"  XLM-R (Próstata):    {test_results['eval_f1']*100:.2f}%")
print(f"  Diferencia:          {improvement:+.2f} puntos")
print(f"\n🚀 Modelo publicado: https://huggingface.co/{HF_MODEL_NAME}")
print(f"\n✅ Archivos guardados en: {OUTPUT_DIR}")
print("="*80)

RESUMEN DEL ENTRENAMIENTO

Modelo: XLM-RoBERTa (xlm-roberta-base)
Dataset: Próstata (NER clínico)
Entidades: 23 etiquetas (10 tipos + O)

Configuración:
  Batch size: 16
  Learning rate: 2e-05
  Épocas entrenadas: 10

🎯 Resultados en Test:
  F1-score:  96.70%
  Precision: 96.41%
  Recall:    97.00%

📊 Comparación:
  BiLSTM (CoNLL-2002): 72.00%
  XLM-R (Próstata):    96.70%
  Diferencia:          +24.70 puntos

🚀 Modelo publicado: https://huggingface.co/Lucyan85/xlmr-ner-prostata

✅ Archivos guardados en: ./xlmr-ner-prostata-batch16
